# Task 01 – Dataset Understanding & Preprocessing
## VisDrone2019 Aerial Imagery Dataset

This notebook covers:
1. Dataset structure overview
2. Annotation format explanation
3. Class distribution & statistics
4. Key challenges
5. Preprocessing & augmentation strategy
6. Sample visualizations

In [ ]:
import sys, os
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

# ── Update this path to match your machine ──
DATA_ROOT  = r'D:\antlings-drone-detection\dataset\VisDrone_Dataset'

TRAIN_SPLIT = 'VisDrone2019-DET-train'
VAL_SPLIT   = 'VisDrone2019-DET-val'
TEST_DEV    = 'VisDrone2019-DET-test-dev'
TEST_CHALL  = 'VisDrone2019-DET-test-challenge'

os.makedirs('../outputs/images', exist_ok=True)

print('Setup complete.')
print(f'Dataset root: {DATA_ROOT}')
print(f'Exists: {Path(DATA_ROOT).exists()}')

## 1. Dataset Structure

```
dataset/
└── VisDrone_Dataset/
    ├── VisDrone2019-DET-train/
    │   ├── images/        # .jpg drone images
    │   └── annotations/   # .txt per-image labels
    ├── VisDrone2019-DET-val/
    │   ├── images/
    │   └── annotations/
    ├── VisDrone2019-DET-test-dev/
    │   └── images/        # no annotations
    ├── VisDrone2019-DET-test-challenge/
    │   └── images/        # no annotations
    └── visdrone.yaml
```

**Annotation format** (comma-separated, one object per line):
```
bbox_left, bbox_top, bbox_width, bbox_height, score, category, truncation, occlusion
```

**Original 10 classes → We keep 2:**
- pedestrian(1) + people(2) → **person**
- car(4) + van(5)           → **car**

In [ ]:
# Count images and annotations per split
splits = [TRAIN_SPLIT, VAL_SPLIT, TEST_DEV, TEST_CHALL]
print(f"{'Split':<40s} {'Images':>8s} {'Annotations':>12s}")
print('-' * 62)

for split in splits:
    img_dir = Path(DATA_ROOT) / split / 'images'
    ann_dir = Path(DATA_ROOT) / split / 'annotations'
    n_imgs = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_anns = len(list(ann_dir.glob('*.txt'))) if ann_dir.exists() else 'N/A'
    print(f"{split:<40s} {n_imgs:>8,} {str(n_anns):>12s}")

In [ ]:
# Analyze class distribution from training set
VISDRONE_CLASSES = {
    0:'ignored', 1:'pedestrian', 2:'people', 3:'bicycle',
    4:'car', 5:'van', 6:'truck', 7:'tricycle',
    8:'awning-tricycle', 9:'bus', 10:'motor'
}

ann_dir = Path(DATA_ROOT) / TRAIN_SPLIT / 'annotations'
class_counts = Counter()
box_widths, box_heights, box_areas = [], [], []
objects_per_image = []

ann_files = sorted(ann_dir.glob('*.txt'))[:1000]

for ann_file in tqdm(ann_files, desc='Parsing annotations'):
    count = 0
    with open(ann_file) as f:
        for line in f:
            p = line.strip().split(',')
            if len(p) < 6: continue
            bw, bh  = int(p[2]), int(p[3])
            score, cat = int(p[4]), int(p[5])
            if score == 0: continue
            class_counts[VISDRONE_CLASSES.get(cat, str(cat))] += 1
            if bw > 0 and bh > 0:
                box_widths.append(bw)
                box_heights.append(bh)
                box_areas.append(bw * bh)
                count += 1
    objects_per_image.append(count)

print('\nClass distribution (1000-image sample):')
for cls, cnt in class_counts.most_common():
    print(f'  {cls:20s}: {cnt:6,d}')
print(f'\nAvg objects per image: {np.mean(objects_per_image):.1f}')

In [ ]:
# Plot statistics
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VisDrone Dataset Statistics (Train – 1k image sample)', fontsize=13, fontweight='bold')

# 1. Class bar chart
sorted_cls = class_counts.most_common()
names  = [c[0] for c in sorted_cls]
counts = [c[1] for c in sorted_cls]
colors = ['#00C853' if n in ('pedestrian','people')
          else '#FF6D00' if n in ('car','van')
          else '#607D8B' for n in names]
axes[0].barh(names, counts, color=colors, edgecolor='none')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()
axes[0].legend(handles=[
    Patch(facecolor='#00C853', label='person (kept)'),
    Patch(facecolor='#FF6D00', label='car (kept)'),
    Patch(facecolor='#607D8B', label='ignored'),
])

# 2. Box size scatter
idx = np.random.choice(len(box_widths), min(3000, len(box_widths)), replace=False)
axes[1].scatter([box_widths[i] for i in idx], [box_heights[i] for i in idx],
                alpha=0.15, s=8, color='#2979FF')
axes[1].set_title('Bounding Box Sizes (px)')
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Height (px)')
axes[1].axvline(32, color='red', linestyle='--', label='32px')
axes[1].axhline(32, color='red', linestyle='--')
axes[1].legend()

# 3. Objects per image
axes[2].hist(objects_per_image, bins=40, color='#AA00FF', edgecolor='none', alpha=0.85)
axes[2].set_title('Objects per Image')
axes[2].set_xlabel('Count')
axes[2].set_ylabel('Frequency')
axes[2].axvline(np.mean(objects_per_image), color='orange', linestyle='--',
                label=f'Mean: {np.mean(objects_per_image):.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../outputs/images/01_dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/01_dataset_analysis.png')

In [ ]:
# Show sample images with ground truth annotations
VISDRONE_TO_OURS = {1: 0, 2: 0, 4: 1, 5: 1}

img_dir     = Path(DATA_ROOT) / TRAIN_SPLIT / 'images'
ann_dir_raw = Path(DATA_ROOT) / TRAIN_SPLIT / 'annotations'
img_files   = sorted(img_dir.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Ground Truth Annotations — person=green, car=orange', fontsize=13, fontweight='bold')

for idx, img_file in enumerate(img_files):
    img = cv2.cvtColor(cv2.imread(str(img_file)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ann_file = ann_dir_raw / img_file.with_suffix('.txt').name
    ax = axes[idx // 3][idx % 3]
    ax.imshow(img)

    person_cnt = car_cnt = 0
    if ann_file.exists():
        with open(ann_file) as f:
            for line in f:
                p = line.strip().split(',')
                if len(p) < 6: continue
                bx, by, bw, bh = int(p[0]), int(p[1]), int(p[2]), int(p[3])
                score, cat = int(p[4]), int(p[5])
                if score == 0 or cat not in VISDRONE_TO_OURS or bw <= 0 or bh <= 0: continue
                cls_id = VISDRONE_TO_OURS[cat]
                color = '#00C853' if cls_id == 0 else '#FF6D00'
                ax.add_patch(patches.Rectangle((bx, by), bw, bh,
                             linewidth=0.8, edgecolor=color, facecolor='none'))
                if cls_id == 0: person_cnt += 1
                else: car_cnt += 1

    ax.set_title(f'Person: {person_cnt}  Car: {car_cnt} | {img_file.name}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/images/01_sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/01_sample_annotations.png')

## 2. Key Challenges

| Challenge | Impact | Mitigation |
|-----------|--------|------------|
| **Tiny objects** | Most bboxes <20px tall | Higher resolution, mosaic augmentation |
| **Dense scenes** | 50-200 objects per image | NMS tuning, copy-paste augmentation |
| **Class imbalance** | Pedestrians 3x more than cars | Focal loss in YOLOv8 |
| **Altitude variance** | Objects change size with altitude | Scale jitter (0.5) |
| **Viewpoint** | Top-down oblique views | Rotation augmentation, fine-tuning |

## 3. Preprocessing & Augmentation

- Annotation conversion: VisDrone CSV → YOLO normalized format
- Mosaic (1.0), MixUp (0.15), Copy-Paste (0.1)
- Scale jitter +/-50%, flip, rotation +/-10 degrees, HSV jitter

### Task 01 Complete
- Dataset root: `VisDrone_Dataset/` with 4 splits
- Keeping 2 classes: person + car
- ~70% of boxes smaller than 32x32 pixels
- Annotations ready for YOLO conversion in Task 02